# 08 · De la secant a la derivada

[Obre aquest quadern a Google Colab](https://colab.research.google.com/github/mlacasa/1BATXILLERAT/blob/main/Derivades_BAT.ipynb)

**Matemàtiques · 1r de batxillerat**  
**Autoria: Dr. Lacasa-Cazcarra · Any 2026**

**Objectiu.** Entendre la derivada com un límit finit de pendents i distingir tres comportaments locals.

**Abans de començar:** quadern 07 i pendent d'una recta  
**Temps orientatiu:** 2 sessions. Hi ha activitats bàsiques i ampliacions opcionals.

**Funcionament.** Executa les cel·les en ordre, des del començament. Primer fes una predicció,
després experimenta i escriu una justificació. Els controls són opcionals: també pots
modificar els arguments de les funcions i executar-les. Les figures representen mostres
finites; les propietats infinites necessiten una demostració.

**Procedència.** Reelaboració de Derivades_BAT.ipynb. Es conserven els originals en un arxiu separat.


In [ ]:
import math
from fractions import Fraction
from decimal import Decimal, localcontext
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "font.size": 11, "figure.dpi": 100})

def explora(funcio, **rangs):
    """Controls opcionals: la mateixa funció també es pot cridar directament."""
    try:
        import ipywidgets as widgets
    except ImportError:
        print("Sense controls: executem els paràmetres per defecte. Pots canviar-los a la crida.")
        return funcio()
    panell = widgets.interactive(funcio, **rangs)
    display(panell)
    return panell


## 1. La tangent no és simplement una recta que toca una corba
Prenem $A=(a,f(a))$ i $B=(a+h,f(a+h))$, amb $h\ne0$.
El pendent de la secant és
$$m_h=\frac{f(a+h)-f(a)}h.$$
Si aquest quocient té un límit **real finit**, el denotem $f'(a)$.
La tangent té equació $y=f(a)+f'(a)(x-a)$.

Per a $f(x)=x^2$, simplifica el quocient abans d'executar:
$$m_h=\frac{(a+h)^2-a^2}h=2a+h\longrightarrow2a.$$


In [ ]:
def funcio(tipus, x):
    if tipus == "quadratica": return np.asarray(x)**2
    if tipus == "absolut": return np.abs(x)
    if tipus == "arrel_cubica": return np.cbrt(x)
    raise ValueError("Funció desconeguda.")

def pendent_secant(tipus, a, h):
    if h == 0: raise ValueError("El pas h ha de ser diferent de zero.")
    if tipus == "quadratica": return 2*a+h  # forma algebraica estable
    return float((funcio(tipus, a+h)-funcio(tipus, a))/h)

def secant(tipus="quadratica", a=0.0, exponent=2, costat=1):
    h = costat * 10.0**(-exponent)
    m = pendent_secant(tipus, a, h)
    x = np.linspace(-2.2, 2.2, 601)
    fa = float(funcio(tipus, a))
    fig, ax = plt.subplots()
    ax.plot(x, funcio(tipus, x), color="#2563eb", label="Funció")
    ax.plot(x, fa+m*(x-a), color="#d97706", label=f"Secant: pendent {m:.5g}")
    ax.scatter([a, a+h], [fa, float(funcio(tipus, a+h))], color="#dc2626")
    if tipus == "quadratica":
        ax.plot(x, fa+2*a*(x-a), "--", color="#16a34a", label=f"Tangent: pendent {2*a:g}")
    ax.set(xlim=(-2.2, 2.2), ylim=(-3, 5), xlabel="x", ylabel="y", title=f"h = {h:g}")
    ax.legend(); plt.show()
    print(f"Quocient incremental = {m:.10g}. Un valor finit de h no és el límit.")
panell = explora(secant, tipus=["quadratica", "absolut", "arrel_cubica"],
                 a=(-1.5, 1.5, .25), exponent=(0, 6, 1), costat=[-1, 1])


## 2. Al punt a=0: tres resultats diferents
* $x^2$: $m_h=h\to0$ pels dos costats. Derivada igual a 0.
* $|x|$: $m_h=1$ per h>0 i $m_h=-1$ per h<0. No hi ha un únic límit.
* $\sqrt[3]{x}$: $m_h=|h|^{-2/3}\to+\infty$. Hi ha tangent vertical,
  però no una derivada real finita a 0.

Les tres funcions són contínues a 0. **Continuïtat no implica derivabilitat.**


In [ ]:
for tipus in ("quadratica", "absolut", "arrel_cubica"):
    print("\n", tipus)
    for h in (0.1, 0.001, -0.001, -0.1):
        print(f"h={h:8g}: pendent={pendent_secant(tipus, 0, h):12.6g}")


## 3. Un pont amb Cauchy, amb una precaució
Si la derivada existeix, els pendents calculats al llarg de qualsevol successió
$h_n\to0$, $h_n\ne0$, convergeixen al mateix nombre i són de Cauchy.
**Una sola successió de pendents de Cauchy no demostra la derivabilitat:**
per a $|x|$, els passos $h_n=1/n$ donen sempre pendent 1, però els passos negatius donen −1.

## 4. Regles i correccions conceptuals
| Funció | Derivada i condicions |
|---|---|
| $x^3$ | $3x^2$ |
| $\sin x$ | $\cos x$, amb angles en radians |
| $\cos x$ | $-\sin x$ |
| $\ln x$ | $1/x$, per a x>0 |
| $g(h(x))$ | $g'(h(x))h'(x)$, quan les derivades necessàries existeixen |

La identitat correcta és $\cos^2x-1=-\sin^2x$.
Si $f'(a)=0$, a no ha de ser un extrem: $f(x)=x^3$ al punt 0 és un contraexemple.
Si $f''(a)=0$, tampoc podem concloure que la funció sigui una recta en un entorn d'a.


In [ ]:
# L'arrodoniment pot malmetre el quocient calculat per una resta de nombres propers.
a = 1.0
print("h          quocient directe       expressió 2a+h")
for h in (1e-2, 1e-6, 1e-10, 1e-14, 1e-17):
    directe = ((a+h)**2-a**2)/h
    print(f"{h:9.1e}  {directe:20.12g}  {2*a+h:20.12g}")


## Activitats
1. Troba l'equació de la tangent de $x^2$ al punt a=1 i al punt a=−1.
2. Explica per què no podem substituir h=0 al quocient incremental.
3. Per a $|x|$, canvia a de 0 a 1. Què passa amb els pendents quan h és petit?
4. **Ampliació:** demostra que ser derivable a a implica ser contínua a a,
   escrivint $f(a+h)-f(a)=h\,[f(a+h)-f(a)]/h$.
5. Escriu una conclusió sobre el recorregut: per què necessitem ℝ abans d'estudiar límits i derivades?

<details><summary>Comprovació</summary>

Les tangents són $y=2x-1$ i $y=-2x-1$. A a=1, el valor absolut coincideix localment
amb x, i la derivada és 1. En la demostració de continuïtat, el quocient té límit
finit i el factor h tendeix a zero. Els experiments numèrics són evidència;
les identitats i els arguments de límits proporcionen les justificacions.
</details>


**Navegació:** [Anterior](07_Limits_Continuitat.ipynb) · [Índex i guia](README.md)
